# Step 2: Data Download + EDA
Chest X-Ray Images (Pneumonia): Kermany et al.

**اختار طريقة واحدة:**
- **Kaggle Notebook (الأسهل، من غير token):** New Notebook ← Add Input ← ابحث `chest-xray-pneumonia` ← فعّل GPU ← شغّل الـ notebook ده.
- **Colab:** شغّل الخلية الخاصة بالتحميل وارفع `kaggle.json` لما تتطلب منك.

In [ ]:
import os, sys, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

ON_KAGGLE = Path("/kaggle/input").exists()
ON_COLAB = "google.colab" in sys.modules
print("Kaggle:", ON_KAGGLE, "| Colab:", ON_COLAB)

WORK = Path("/kaggle/working") if ON_KAGGLE else Path("/content") if ON_COLAB else Path(".")
FIG_DIR = WORK / "outputs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
random.seed(SEED); np.random.seed(SEED)

## 1) تحميل البيانات

In [ ]:
def find_root(base):
    """Return the folder that contains train/ and test/."""
    for p in Path(base).rglob("train"):
        if p.is_dir() and (p.parent / "test").exists():
            return p.parent
    return None

DATA_ROOT = None
if ON_KAGGLE:
    DATA_ROOT = find_root("/kaggle/input")
else:
    DATA_ROOT = find_root(WORK / "data" / "raw")

print("Found:", DATA_ROOT)

In [ ]:
# Colab / local only: skip this cell on Kaggle or if DATA_ROOT was found above.
if DATA_ROOT is None:
    if ON_COLAB:
        from google.colab import files
        print("ارفع ملف kaggle.json")
        files.upload()
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    assert Path("kaggle.json").exists(), "kaggle.json not found in the current folder"
    os.replace("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    raw = WORK / "data" / "raw"
    raw.mkdir(parents=True, exist_ok=True)
    os.system(f"pip -q install kaggle")
    os.system(f"kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p {raw} --unzip")
    DATA_ROOT = find_root(raw)
    print("Downloaded to:", DATA_ROOT)

assert DATA_ROOT is not None, "Dataset not found. Check the steps in the first cell."

## 2) عدد الصور في كل split وكل class

In [ ]:
EXTS = {".jpeg", ".jpg", ".png"}
rows = []
for split in ["train", "val", "test"]:
    for cls in ["NORMAL", "PNEUMONIA"]:
        for f in (DATA_ROOT / split / cls).iterdir():
            if f.suffix.lower() in EXTS:
                rows.append({"split": split, "label": cls, "path": str(f)})
df = pd.DataFrame(rows)
counts = df.groupby(["split", "label"]).size().unstack(fill_value=0)
counts["total"] = counts.sum(axis=1)
counts

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
counts[["NORMAL", "PNEUMONIA"]].plot(kind="bar", ax=ax, color=["#4C78A8", "#E45756"])
ax.set_title("Images per split and class")
ax.set_ylabel("count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIG_DIR / "class_distribution.png", dpi=150)
plt.show()

train = df[df.split == "train"]
ratio = (train.label == "PNEUMONIA").mean()
print(f"Pneumonia share in train: {ratio:.1%}  ->  imbalanced, we will handle it in step 3/5")

## 3) عينات من الصور

In [ ]:
def show_samples(label, n=6):
    paths = df[(df.split == "train") & (df.label == label)].sample(n, random_state=SEED).path.tolist()
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3.2))
    for a, p in zip(axes, paths):
        a.imshow(Image.open(p).convert("L"), cmap="gray")
        a.axis("off")
    fig.suptitle(label, fontsize=14)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"samples_{label.lower()}.png", dpi=120)
    plt.show()

show_samples("NORMAL")
show_samples("PNEUMONIA")

## 4) أحجام الصور وشدة الإضاءة

In [ ]:
sample = df.sample(min(600, len(df)), random_state=SEED)
stats = []
for p, lab in zip(sample.path, sample.label):
    im = Image.open(p)
    arr = np.asarray(im.convert("L"), dtype=np.float32)
    stats.append({"w": im.width, "h": im.height, "mode": im.mode,
                  "mean": arr.mean(), "std": arr.std(), "label": lab})
stats = pd.DataFrame(stats)
print(stats[["w", "h", "mean", "std"]].describe().round(1))
print("\nImage modes:", stats["mode"].value_counts().to_dict())

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
sns.histplot(data=stats, x="w", hue="label", bins=30, ax=axes[0]); axes[0].set_title("Width")
sns.histplot(data=stats, x="mean", hue="label", bins=30, ax=axes[1]); axes[1].set_title("Mean intensity")
plt.tight_layout()
plt.savefig(FIG_DIR / "size_intensity.png", dpi=150)
plt.show()

## 5) فحص جودة البيانات

In [ ]:
bad = []
for p in df.sample(min(1500, len(df)), random_state=SEED).path:
    try:
        with Image.open(p) as im:
            im.verify()
    except Exception as e:
        bad.append(p)
print("Corrupted files in checked sample:", len(bad))
print("Val split size:", int((df.split == 'val').sum()), "(too small, we re-split in step 3)")

## 6) ملاحظات الـ EDA (هتتحط في التوثيق النهائي)
- الداتا غير متوازنة: Pneumonia حوالي 3x الـ Normal في train، فهنستخدم class weights.
- الـ val الأصلي صغير جدًا (16 صورة)، فهنعيد التقسيم من train+val بنسب 70/15/15 مع تثبيت الـ seed.
- الصور بأحجام مختلفة وكلها تقريبًا grayscale، فهنعمل resize لـ 224 وتحويل لـ 3 قنوات.
- ملاحظة مهمة: لازم نتأكد إن الـ split مفيهوش تسريب (نفس المريض في train و test)، ودي نقطة نذكرها كقيد في التقرير.

In [ ]:
df.drop(columns="path").to_csv(FIG_DIR / "dataset_counts_raw.csv", index=False)
counts.to_csv(FIG_DIR / "counts_summary.csv")
print("Saved EDA outputs to", FIG_DIR)